# DINOv2 embeddings for the soil photos

Extracts pretrained DINOv2 embeddings for all 162 competition photos and saves them to
`/kaggle/working/dinov2_embeddings.csv`. Download that file and send it back for evaluation.

**Before running:** Settings → turn **Internet on** (the model is downloaded from Hugging Face) and, ideally, choose a
**GPU** accelerator. Add the competition data with "+ Add Input" → Competitions → *soil-grain-size-from-photos*.
Run time: about 2 minutes on a GPU, about 10–15 minutes on CPU.

Each photo is first put on the same physical scale as the rest of the pipeline (centre 60 % crop, resampled to
4 px/mm using the camera's pixels-per-mm), so a feature means the same grain size whatever phone took the photo.
Two embeddings are taken per photo, each with a horizontal-flip average:

* **tiles** – 224 × 224 px tiles (56 mm × 56 mm of soil) on a grid, averaged;
* **whole** – the whole canonical crop in one pass (DINOv2 accepts any multiple of 14 px).

For each, both the CLS token and the mean of the patch tokens are saved.

In [ ]:
import glob, os, re, time
import numpy as np, pandas as pd, torch
from PIL import Image, ImageOps
from transformers import AutoModel

MODEL = "facebook/dinov2-base"      # 768-d; "facebook/dinov2-small" (384-d) also works
CANON_PPM, CROP, TILE = 4.0, 0.6, 224
OUT = "/kaggle/working/dinov2_embeddings.csv"

hits = glob.glob("/kaggle/input/**/Training_labels_updated.csv", recursive=True)
assert hits, "Competition data not found: add soil-grain-size-from-photos as an input"
ROOT = os.path.dirname(hits[0])
device = "cuda" if torch.cuda.is_available() else "cpu"
print("data:", ROOT, "| device:", device)

In [ ]:
# Same camera table and file-name parsing as src/gsd.py (effective ppm = ppm * actual_side / reference_side,
# because the host downscaled the training JPGs).
PHONES = {"Motorola_Edge_60_fusion": (4096, 12.465), "Motorola_Edge": (4000, 11.492), "Samsung_A52": (9248, 26.33),
          "iPhone14": (4032, 13.942), "iPhone16": (5712, 19.525), "iPhone_16": (5712, 19.525)}
TEST_ID_FIX = {"HPC_Münster_BS6_9,0-10m": "HPC_Muenster_BS6_9_0-10m"}

def phone_of(fname):
    return next(k for k in PHONES if fname.startswith(k))

def sample_of(fname, split):
    base = os.path.splitext(fname)[0]
    if split == "train":
        return re.search(r"_([A-Z]\d{3})_\d+$", base).group(1)
    s = re.sub(r"\s*\(\d+\)$", "", re.sub(r"^iPhone_?\d+_", "", base))
    return TEST_ID_FIX.get(s, s)

rows = []
for split, d in [("train", "Training-All_Photos_updated"), ("test", "Test_All_Photos")]:
    for f in sorted(glob.glob(os.path.join(ROOT, d, "**", "*.*"), recursive=True)):
        if f.lower().endswith((".jpg", ".jpeg")):
            n = os.path.basename(f)
            rows.append(dict(path=f, file=n, split=split, sample_id=sample_of(n, split), phone=phone_of(n)))
imgs = pd.DataFrame(rows)
print(imgs.groupby("split").size().to_dict(), "| samples:", imgs.groupby("split").sample_id.nunique().to_dict())
assert len(imgs) == 162, len(imgs)

In [ ]:
def load_canonical(path, phone):
    im = ImageOps.exif_transpose(Image.open(path)).convert("RGB")
    ref_side, ppm = PHONES[phone]
    s = CANON_PPM / (ppm * max(im.size) / ref_side)
    w, h = im.size
    cw, ch = int(w * CROP), int(h * CROP)
    im = im.crop(((w - cw) // 2, (h - ch) // 2, (w - cw) // 2 + cw, (h - ch) // 2 + ch))
    im = im.resize((max(TILE, round(cw * s)), max(TILE, round(ch * s))), Image.LANCZOS if s < 1 else Image.BICUBIC)
    return np.asarray(im, dtype=np.float32) / 255.0

MEAN, STD = np.array([0.485, 0.456, 0.406], np.float32), np.array([0.229, 0.224, 0.225], np.float32)
def to_tensor(a):
    return torch.from_numpy(((a - MEAN) / STD).transpose(2, 0, 1)).float()

model = AutoModel.from_pretrained(MODEL).to(device).eval()

@torch.no_grad()
def embed(batch):
    """CLS and mean-patch embeddings, averaged over the image and its horizontal flip."""
    out = []
    for x in (batch, torch.flip(batch, dims=[3])):
        h = model(pixel_values=x.to(device)).last_hidden_state
        out.append(torch.cat([h[:, 0], h[:, 1:].mean(1)], dim=1))
    return ((out[0] + out[1]) / 2).cpu().numpy()

def photo_embedding(path, phone):
    a = load_canonical(path, phone)
    H, W = a.shape[:2]
    ny, nx = max(1, H // TILE), max(1, W // TILE)
    y0, x0 = (H - ny * TILE) // 2, (W - nx * TILE) // 2
    tiles = torch.stack([to_tensor(a[y0 + i * TILE:y0 + (i + 1) * TILE, x0 + j * TILE:x0 + (j + 1) * TILE])
                         for i in range(ny) for j in range(nx)])
    t = embed(tiles).mean(0)
    h14, w14 = (H // 14) * 14, (W // 14) * 14
    g = embed(to_tensor(a[(H - h14) // 2:(H - h14) // 2 + h14, (W - w14) // 2:(W - w14) // 2 + w14])[None])[0]
    return t, g, ny * nx

In [ ]:
t0 = time.time(); T, G, NT = [], [], []
for i, r in imgs.iterrows():
    t, g, n = photo_embedding(r.path, r.phone)
    T.append(t); G.append(g); NT.append(n)
    if i % 20 == 0:
        print(f"{i + 1}/{len(imgs)}  {time.time() - t0:.0f}s")
d = T[0].shape[0] // 2
cols = ([f"tile_cls_{k}" for k in range(d)] + [f"tile_mean_{k}" for k in range(d)] +
        [f"whole_cls_{k}" for k in range(d)] + [f"whole_mean_{k}" for k in range(d)])
emb = pd.DataFrame(np.hstack([np.array(T), np.array(G)]), columns=cols).astype(np.float32)
out = pd.concat([imgs.drop(columns="path").reset_index(drop=True), pd.Series(NT, name="n_tiles"), emb], axis=1)
out.to_csv(OUT, index=False, float_format="%.5f")
print(out.shape, f"{time.time() - t0:.0f}s ->", OUT)
print(out.groupby("phone").n_tiles.describe()[["mean", "min", "max"]])

Download `dinov2_embeddings.csv` from the **Output** panel (right sidebar → `/kaggle/working`) and send it back.
It contains no predictions and uses no labels; it is only image features.